## Import thư viện

In [1]:
import json
import re
import pandas as pd

from tqdm import tqdm
from underthesea import word_tokenize

## 2. Đọc dữ liệu gốc

In [2]:
DATA_PATH = "../data/raw/news_dataset.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Shape:", df.shape)
df.head()

Shape: (184539, 10)


,id,author,content,picture_count,processed,source,title,topic,url,crawled_at
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,2022-08-01 09:09:22.817308
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,2022-08-01 09:09:21.181469
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,2022-08-01 09:09:15.311901
3,218267,Ngọc Ánh,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,1,0,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,2022-08-01 09:09:02.211498
4,218266,HẢI YẾN - MINH LÝ,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,12,0,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,2022-08-01 09:09:01.601170


## 3. Kiểm tra thông tin dữ liệu

In [3]:
df.columns

Index(['id', 'author', 'content', 'picture_count', 'processed', 'source',
       'title', 'topic', 'url', 'crawled_at'],
      dtype='str')

## 4. Chọn các cột cần thiết

In [4]:
required_columns = ["id", "title", "content", "topic", "source", "url"]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

df["title"] = df["title"].fillna("")
df["content"] = df["content"].fillna("")
df["topic"] = df["topic"].fillna("")
df["source"] = df["source"].fillna("")
df["url"] = df["url"].fillna("")

df["raw_text"] = (
    (df["title"] + " ") * 3 +
    df["content"]
)

df = df[df["raw_text"].str.strip() != ""].reset_index(drop=True)

print("Shape after removing empty text:", df.shape)

df[["id", "title", "topic", "source", "raw_text"]].head()

Shape after removing empty text: (184521, 11)


,id,title,topic,source,raw_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ..."
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G Bỏ..."
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên..."


## 5. Xóa dữ liệu trùng lặp

In [5]:
before = len(df)

df = df.drop_duplicates(subset=["title", "content"]).reset_index(drop=True)

after = len(df)

print("Before:", before)
print("After:", after)
print("Removed duplicates:", before - after)

Before: 184521
After: 182568
Removed duplicates: 1953


## 6. Xây dựng hàm tiền xử lý văn bản

In [6]:
vietnamese_stopwords = set([
    "là", "và", "của", "có", "cho", "với", "trong", "khi",
    "những", "các", "một", "được", "đã", "đang", "này",
    "đó", "thì", "mà", "ở", "về", "từ", "đến", "theo",
    "sau", "trước", "trên", "dưới", "vào", "ra", "năm",
    "ngày", "tháng", "cũng", "như", "nếu", "vì", "do",
    "để", "bị", "tại", "nên", "sẽ", "rằng", "nhiều"
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in vietnamese_stopwords]
    return " ".join(tokens)

def preprocess(text):
    text = clean_text(text)
    text = word_tokenize(text, format="text")
    text = remove_stopwords(text)
    return text

## 7. Kiểm tra thử hàm tiền xử lý

In [7]:
sample_text = df.loc[0, "raw_text"]

print("Before:")
print(sample_text[:700])

print("\nAfter:")
print(preprocess(sample_text)[:700])

Before:
Tên cướp tiệm vàng tại Huế là đại uý công an, công tác tại Trại giam Tên cướp tiệm vàng tại Huế là đại uý công an, công tác tại Trại giam Tên cướp tiệm vàng tại Huế là đại uý công an, công tác tại Trại giam Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã có thông tin ban đầu về vụ nổ súng,cướp tiệm vàng tại chợ Đông Ba nằm trên đường Trần Hưng Đạo (TP Huế, tỉnh Thừa Thiên - Huế). Thông Sài Gòn Giải Phóng, khoảng 12h30' ngày 31/7, một đối tượng sử dụng súng AK bất ngờ xông vào tiệm vàng Hoàng Đức và Thái Lợi (phía trước chợ Đông Ba) rồi nổ súng chỉ thiên liên tiếp uy hiếp chủ tiệm để cướp vàng. Sau đó, đối tượng mang số vàng vừa cướp được vứt ra vỉa hè rồi đi bộ đến khu vực cầu Gia Hội, cách khu

After:
tên cướp tiệm vàng huế đại_úy công_an công_tác trại_giam tên cướp tiệm vàng huế đại_úy công_an công_tác trại_giam tên cướp tiệm vàng huế đại_úy công_an công_tác trại_giam chiều 31 7 công_an tỉnh thừa thiên_huế thông_tin ban_đầu vụ nổ_súng cướp tiệm vàng chợ đông ba nằm đường trần_hư

## 8. Áp dụng tiền xử lý cho toàn bộ dataset

In [8]:
tqdm.pandas()

df["processed_text"] = df["raw_text"].progress_apply(preprocess)

df = df[df["processed_text"].str.strip() != ""].reset_index(drop=True)

print("Shape after preprocessing:", df.shape)

df[["id", "title", "topic", "processed_text"]].head()

100%|██████████| 182568/182568 [1:20:16<00:00, 37.90it/s]  


Shape after preprocessing: (182566, 12)


,id,title,topic,processed_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,người chết mưa_lũ nghìn mỹ tăng lên 28 người c...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 9. Lưu dữ liệu đã xử lý

In [9]:
OUTPUT_PATH = "../data/processed/news_processed.pkl"

df.to_pickle(OUTPUT_PATH)

print("Saved processed data to:", OUTPUT_PATH)

Saved processed data to: ../data/processed/news_processed.pkl
